In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams['font.sans-serif'] = ['SimHei'] 
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white", font='SimHei')

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 DWD 表中有数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def fetch_house_orientation_data(city_name):
    """
    获取指定城市的户型、朝向与总价数据。
    """
    conn = get_db_connection()
    query = f"""
    SELECT 
        concat(room_num, '室', hall_num, '厅', bathroom_num, '卫') as room_type,
        CASE 
            WHEN orientation LIKE '%南%' THEN '朝南' 
            ELSE '非朝南' 
        END as orientation_group,
        total_price
    FROM iceberg_catalog.ershoufang.dwd_ershoufang_fact 
    WHERE city = '{city_name}' 
      AND room_num IS NOT NULL 
      AND hall_num IS NOT NULL
      AND bathroom_num IS NOT NULL
      AND total_price IS NOT NULL
      AND total_price > 0
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def plot_mirrored_barh(city):
    """绘制水平镜像条形图 (蝴蝶图)"""
    df = fetch_house_orientation_data(city)
    
    if df.empty or len(df) < 50:
        plt.figure(figsize=(10, 6))
        plt.text(0.5, 0.5, f"{city} 有效数据不足，无法生成镜像对比图", ha='center', va='center', fontsize=14)
        plt.axis('off')
        plt.show()
        return

    # --- 1. 数据预处理 ---
    # 筛选出该城市数量最多的 TOP 10 热门户型进行对比
    top_10_types = df['room_type'].value_counts().nlargest(10).index.tolist()
    plot_df = df[df['room_type'].isin(top_10_types)]

    # 计算各个户型在两个朝向下的“总价中位数”
    pivot_df = plot_df.groupby(['room_type', 'orientation_group'])['total_price'].median().unstack(fill_value=0)
    
    # 确保列存在，防止某些城市完全没有朝南或非朝南的房子导致报错
    if '朝南' not in pivot_df.columns: pivot_df['朝南'] = 0
    if '非朝南' not in pivot_df.columns: pivot_df['非朝南'] = 0

    # 按照整体的中位数（朝南+非朝南）进行升序排列
    pivot_df['total_median'] = pivot_df['朝南'] + pivot_df['非朝南']
    pivot_df = pivot_df.sort_values(by='total_median', ascending=True)

    # --- 2. 准备绘图数据 (镜像转换核心) ---
    y_labels = pivot_df.index.tolist()
    y_pos = np.arange(len(y_labels))

    val_left = -pivot_df['非朝南'].values
    val_right = pivot_df['朝南'].values

    # --- 3. 开始绘图 ---
    fig, ax = plt.subplots(figsize=(14, 8))

    color_left = '#5DADE2'
    color_right = '#F5B041' 

    # 绘制左右两侧的水平条形图
    ax.barh(y_pos, val_left, color=color_left, edgecolor='white', height=0.6, label='非朝南 (无南向)')
    ax.barh(y_pos, val_right, color=color_right, edgecolor='white', height=0.6, label='朝南 (包含南向)')

    # --- 4. 标注与修饰 ---
    for i in range(len(y_labels)):
        # 左侧数值 (非朝南)
        if val_left[i] != 0:
            ax.text(val_left[i] - (abs(val_left).max() * 0.02), y_pos[i], 
                    f"{abs(val_left[i]):.1f}", ha='right', va='center', color='#2C3E50', fontsize=11)
        # 右侧数值 (朝南)
        if val_right[i] != 0:
            ax.text(val_right[i] + (val_right.max() * 0.02), y_pos[i], 
                    f"{val_right[i]:.1f}", ha='left', va='center', color='#2C3E50', fontsize=11)

    # 绘制中间的 0 刻度基准线，强化对称感
    ax.axvline(0, color='#333333', linewidth=1.5, zorder=3)

    plt.title(f'[{city}] TOP 10热门户型中位数总价对比：朝南 vs 非朝南 (万)', fontsize=16, pad=20)
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f"{abs(int(x))}"))
    plt.xlabel('中位数总价 (万元)', fontsize=12, labelpad=10)
    
    # 设置 Y 轴标签
    ax.set_yticks(y_pos)
    ax.set_yticklabels(y_labels, fontsize=12)
    
    # 优化图例和网格
    ax.legend(loc='upper right', bbox_to_anchor=(0.98, 0.2), frameon=True, fontsize=11)
    ax.grid(axis='x', linestyle='--', alpha=0.4)
    sns.despine(left=True, top=True, right=True)

    # 动态留出左右两边的边距，防止数值文本被裁剪
    max_range = max(abs(val_left).max(), val_right.max()) * 1.15
    ax.set_xlim(-max_range, max_range)

    plt.tight_layout()
    plt.show()

In [3]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                plot_mirrored_barh(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        plot_mirrored_barh(default_city)

In [4]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=4, layout=Layout(width='250px'), options=('上海', '东莞', '中山', '佛山', '北京', …

Output()